In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd


path = '/content/drive/MyDrive/DataAI/'

df_train = pd.read_csv(path + 'train_processed.csv')
df_val = pd.read_csv(path + 'val_processed.csv')
df_test = pd.read_csv(path + 'test_processed.csv')


print(df_train.head())

                                             cmt_col  labels  length  \
0                                                cặc     1.0       3   
1                   đr hết thời cày rank bk còn ngợp     0.0      32   
2         Diễn viên hô ni hút mà ngại gì mấy khác kk     0.0      42   
3  Toàn bọn vô công rồi nghề ngồi sủa ăn lương th...     2.0     228   
4  lên kim cương bố ỉa vào mồm mày với cái lối lê...     1.0     121   

                                      processed_text  
0                                                cặc  
1                   đr hết thời cày rank bk còn ngợp  
2         diễn_viên hô ni hút mà ngại gì mấy khác kk  
3  toàn bọn vô công rồi nghề ngồi sủa ăn lương th...  
4  lên kim_cương bố ỉa vào mồm mày với cái lối lê...  


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib

# ==========================================
# 1. TÁCH DỮ LIỆU (ĐÃ SỬA TÊN CỘT CHUẨN)
# ==========================================
text_col = 'processed_text' # Cột chứa chữ
label_col = 'labels'        # Cột chứa nhãn

X_train = df_train[text_col].fillna("").astype(str)
y_train = df_train[label_col]

X_val = df_val[text_col].fillna("").astype(str)
y_val = df_val[label_col]

X_test = df_test[text_col].fillna("").astype(str)
y_test = df_test[label_col]

# ==========================================
# 2. VECTOR HÓA BẰNG TF-IDF
# ==========================================
print("Đang chuyển đổi văn bản thành vector...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Kích thước ma trận TF-IDF tập Train:", X_train_tfidf.shape)

# ==========================================
# 3. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH
# ==========================================
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    print(f"\n========== KẾT QUẢ MÔ HÌNH: {model_name} ==========")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score : {f1_score(y_test, y_pred, average='macro'):.4f}")
    print("\nChi tiết Classification Report:\n")
    print(classification_report(y_test, y_pred))
    print("="*55)

# Huấn luyện Logistic Regression
print("\nĐang huấn luyện Logistic Regression...")
logreg_model = LogisticRegression(max_iter=1000, random_state=42)
logreg_model.fit(X_train_tfidf, y_train)

# Huấn luyện SVM
print("Đang huấn luyện SVM (có thể mất vài phút)...")
svm_model = SVC(kernel='linear', random_state=42, probability=True) # Thêm probability=True để vẽ biểu đồ và xuất % độ tin cậy sau này
svm_model.fit(X_train_tfidf, y_train)

# Đánh giá trên tập Test
evaluate_model(logreg_model, X_test_tfidf, y_test, "Logistic Regression")
evaluate_model(svm_model, X_test_tfidf, y_test, "SVM")

# ==========================================
# 4. LƯU MÔ HÌNH VỀ GOOGLE DRIVE
# ==========================================
# Nhớ sửa lại đường dẫn này khớp với thư mục của bạn trên Drive
save_path = '/content/drive/MyDrive/DataAI/'

joblib.dump(vectorizer, save_path + 'tfidf_vectorizer.pkl')
joblib.dump(logreg_model, save_path + 'logreg_model.pkl')
joblib.dump(svm_model, save_path + 'svm_model.pkl')

print(f"\n✅ Đã lưu thành công 3 file .pkl vào: {save_path}")

Đang chuyển đổi văn bản thành vector...
Kích thước ma trận TF-IDF tập Train: (5166, 10000)

Đang huấn luyện Logistic Regression...
Đang huấn luyện SVM (có thể mất vài phút)...

========== KẾT QUẢ MÔ HÌNH: Logistic Regression ==========
Accuracy : 0.7453
Precision: 0.7432
Recall   : 0.7339
F1-score : 0.7369

Chi tiết Classification Report:

              precision    recall  f1-score   support

         0.0       0.77      0.86      0.82       600
         1.0       0.68      0.63      0.66       496
         2.0       0.77      0.71      0.74       380

    accuracy                           0.75      1476
   macro avg       0.74      0.73      0.74      1476
weighted avg       0.74      0.75      0.74      1476


========== KẾT QUẢ MÔ HÌNH: SVM ==========
Accuracy : 0.7520
Precision: 0.7471
Recall   : 0.7440
F1-score : 0.7450

Chi tiết Classification Report:

              precision    recall  f1-score   support

         0.0       0.79      0.85      0.82       600
         1.0      